# Nifty Weekly Options — Strategy Analysis 2024–2026 (Corrected)

**10 deployed strategies | Fixed SL/TP matching live paper_trader.py | Realistic compounding**

**Data:** Jan 2024–Mar 2026 (old monthly format Jan–Sep 2024; per-strike expiry folders Oct 2024+)

**Fixes applied vs original version:**
1. **Sign bug fixed** — `pnl_arr = nc − (piv @ wt)` is actual PnL; was inverted before
2. **Tue–Fri entries only** — matches live cron schedule (Mon excluded)
3. **Kite 09:25 ATM** — uses kite minute cache as primary source; nifty spot as fallback
4. **No in-sample grid search** — fixed SL/TP params taken directly from `paper_trader.py`

**Realism features (unchanged):**
- Real NIFTY options tick data (open prices, bar-by-bar scanning)
- Bid-ask slippage every leg, entry + exit (×2 round-trip)
- Real broker charges (brokerage, STT, exchange, GST, SEBI)
- Per-trade lot sizing: `lots = min(MAX_LOTS, floor(capital / (SPAN × buffer)))`
- 15% account drawdown circuit-breaker → pause 4 weeks, reset peak
- Monthly lot recalculation only


In [ ]:
STARTING_CAPITAL = 100_000

SLIP_PER_UNIT = 8.0    # Rs per unit per leg, round-trip (×2 applied in simulate)
MARGIN_BUF    = 2.0
DD_LIMIT      = 0.15
PAUSE_WEEKS   = 4

LOT_SIZE    = 75
STRIKE_STEP = 50
ENTRY_TIME  = '09:25'
EXIT_TIME   = '15:20'

# Entry days matching live cron (Tue-Fri only; Mon excluded)
VALID_ENTRY_WDAYS = {1, 2, 3, 4}

SPREAD_W    = 4
BAT_MID_W   = 1; BAT_OUTER_W = 3; BAT_FAR_W = 6

SPAN = {
    'Bear Call Spread': 25_000, 'Bull Put Spread': 25_000,
    'Batman': 90_000, 'Strip': 25_000, 'Long Straddle': 20_000,
}
MAX_LOTS = {
    'Bear Call Spread': 5, 'Bull Put Spread': 5,
    'Batman': 1, 'Strip': 5, 'Long Straddle': 5,
}

NSE_HOLIDAYS = {
    # 2024
    __import__('datetime').date(2024, 1,22), __import__('datetime').date(2024, 3,25),
    __import__('datetime').date(2024, 3,29), __import__('datetime').date(2024, 4,14),
    __import__('datetime').date(2024, 4,17), __import__('datetime').date(2024, 5,23),
    __import__('datetime').date(2024, 6,17), __import__('datetime').date(2024, 7,17),
    __import__('datetime').date(2024, 8,15), __import__('datetime').date(2024,10, 2),
    __import__('datetime').date(2024,10,24), __import__('datetime').date(2024,11, 1),
    __import__('datetime').date(2024,11,15), __import__('datetime').date(2024,12,25),
    # 2025
    __import__('datetime').date(2025, 2,26), __import__('datetime').date(2025, 3,14),
    __import__('datetime').date(2025, 3,31), __import__('datetime').date(2025, 4,10),
    __import__('datetime').date(2025, 4,14), __import__('datetime').date(2025, 4,18),
    __import__('datetime').date(2025, 5, 1), __import__('datetime').date(2025, 8,15),
    __import__('datetime').date(2025, 8,27), __import__('datetime').date(2025,10, 2),
    __import__('datetime').date(2025,10,21), __import__('datetime').date(2025,10,22),
    __import__('datetime').date(2025,11, 5), __import__('datetime').date(2025,12,25),
    # 2026
    __import__('datetime').date(2026, 1,26), __import__('datetime').date(2026, 3,26),
}

# 10 deployed strategies — params from paper_trader.py (fixed, not searched)
DEPLOYED = [
    {'label': 'Bear Call Spread DTE1', 'sname': 'Bear Call Spread', 'mod_key': 'bcs', 'dte': 1, 'sl': 0.90, 'tp': 0.80},
    {'label': 'Bear Call Spread DTE2', 'sname': 'Bear Call Spread', 'mod_key': 'bcs', 'dte': 2, 'sl': 1.00, 'tp': 0.80},
    {'label': 'Bear Call Spread DTE3', 'sname': 'Bear Call Spread', 'mod_key': 'bcs', 'dte': 3, 'sl': 1.00, 'tp': 0.80},
    {'label': 'Bear Call Spread DTE4', 'sname': 'Bear Call Spread', 'mod_key': 'bcs', 'dte': 4, 'sl': 1.00, 'tp': 0.80},
    {'label': 'Bull Put Spread DTE2',  'sname': 'Bull Put Spread',  'mod_key': 'bps', 'dte': 2, 'sl': 0.65, 'tp': 0.80},
    {'label': 'Bull Put Spread DTE3',  'sname': 'Bull Put Spread',  'mod_key': 'bps', 'dte': 3, 'sl': 0.75, 'tp': 0.80},
    {'label': 'Bull Put Spread DTE4',  'sname': 'Bull Put Spread',  'mod_key': 'bps', 'dte': 4, 'sl': 0.75, 'tp': 0.80},
    {'label': 'Strip DTE4',            'sname': 'Strip',            'mod_key': 'strip', 'dte': 4, 'sl': 1.00, 'tp': 0.25},
    {'label': 'Batman DTE2',           'sname': 'Batman',           'mod_key': 'batman', 'dte': 2, 'sl': 0.90, 'tp': 0.80},
    {'label': 'Long Straddle DTE1',    'sname': 'Long Straddle',    'mod_key': 'straddle', 'dte': 1, 'sl': 0.90, 'tp': 0.35},
]

print("Config loaded.")
print(f"  Entry days: Tue-Fri only (weekdays {sorted(VALID_ENTRY_WDAYS)})")
print(f"  Strategies: {len(DEPLOYED)}")


In [ ]:
import sys, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

import strategies.bear_call_spread as bcs_mod
import strategies.bull_put_spread  as bps_mod
import strategies.batman           as batman_mod
import strategies.strip            as strip_mod
import strategies.long_straddle    as straddle_mod

MOD_MAP = {'bcs': bcs_mod, 'bps': bps_mod, 'batman': batman_mod,
           'strip': strip_mod, 'straddle': straddle_mod}

from data.data_config import (load_all_expiry_dates, load_option_file,
                              load_nifty_spot, generate_trading_days)
from datetime import date as _dt_date

expiry_dates = [e for e in load_all_expiry_dates()
                if e.weekday() < 5 and e not in NSE_HOLIDAYS]

# ── Trading day calendar: generated from holiday set (covers Jan 2024 – Mar 2026) ──
_BACKTEST_START  = _dt_date(2024, 1, 2)
_BACKTEST_END    = _dt_date(2026, 3, 31)
_sorted_days     = generate_trading_days(_BACKTEST_START, _BACKTEST_END, NSE_HOLIDAYS)
all_trading_days = set(_sorted_days)

# ── NIFTY 1-min spot (2024 only) — fallback ATM source ────────────────────────
ns = load_nifty_spot(2024)
ns['date']     = ns['datetime'].dt.date
ns['time_str'] = ns['datetime'].dt.strftime('%H:%M')
_spot_idx    = ns.set_index(['date', 'time_str'])['open']

def get_spot_fallback(d, t):
    try:    return float(_spot_idx.loc[(d, t)])
    except: return None

# ── Kite minute cache — primary 09:25 ATM source (matches live paper_trader.py) ──
_HERE = Path('.').resolve()
KITE_DIR = _HERE.parent.parent / 'gap_trading' / 'kite_minute_cache'
kite_925_map = {}
if KITE_DIR.exists():
    kite_dfs = []
    for fp in sorted(KITE_DIR.glob('minute_256265_*.pkl')):
        with open(fp, 'rb') as fh:
            kite_dfs.append(pickle.load(fh))
    if kite_dfs:
        nifty_min = pd.concat(kite_dfs).sort_index()
        if nifty_min.index.tz is not None:
            nifty_min.index = nifty_min.index.tz_localize(None)
        for dt, row in nifty_min.iterrows():
            if dt.strftime('%H:%M') == '09:25':
                kite_925_map[dt.date()] = float(row['open'])
        print(f"Kite cache loaded: {len(kite_925_map)} dates with 09:25 NIFTY open")
else:
    print("WARNING: kite_minute_cache not found — using nifty spot fallback for all dates")

def get_nifty_925(d):
    if d in kite_925_map:
        return kite_925_map[d]
    return get_spot_fallback(d, '09:25') or get_spot_fallback(d, '09:15')

# ── Options data cache ────────────────────────────────────────────────────────
_opt_cache = {}
def load_opt_cached(td, exp):
    k = (td, exp)
    if k not in _opt_cache:
        df = load_option_file(trade_date=td, expiry_date=exp)
        if df is not None and 'time_str' not in df.columns:
            df['time_str'] = df['datetime'].astype(str).str[:5]
        _opt_cache[k] = df
    return _opt_cache[k]

def get_n_before(exp, n):
    idx = _sorted_days.index(exp)
    return _sorted_days[idx - n] if idx >= n else None

def get_window(s, e):
    return [d for d in _sorted_days if s <= d <= e]

print(f"Data ready.  Expiries={len(expiry_dates)}  TradingDays={len(all_trading_days)}")
print(f"  Period: {_sorted_days[0]} to {_sorted_days[-1]}")


In [ ]:
def get_legs(strat_mod, atm):
    if strat_mod is bcs_mod:      return strat_mod.get_legs(atm, STRIKE_STEP, lots=1, width=SPREAD_W)
    if strat_mod is bps_mod:      return strat_mod.get_legs(atm, STRIKE_STEP, lots=1, width=SPREAD_W)
    if strat_mod is batman_mod:   return strat_mod.get_legs(atm, STRIKE_STEP, lots=1,
                                      mid_width=BAT_MID_W, outer_width=BAT_OUTER_W, far_width=BAT_FAR_W)
    if strat_mod is strip_mod:    return strat_mod.get_legs(atm, STRIKE_STEP, lots=1)
    if strat_mod is straddle_mod: return strat_mod.get_legs(atm, STRIKE_STEP, lots=1)

def compute_max_loss(legs, entry_px):
    strikes = [l.strike for l in legs]
    spots   = np.arange(min(strikes) - 10*STRIKE_STEP, max(strikes) + 11*STRIKE_STEP, STRIKE_STEP)
    worst   = 0.0
    for s in spots:
        pnl = sum(((entry_px[(l.strike, l.right)] -
                    (max(0, s - l.strike) if l.right == 'CE' else max(0, l.strike - s)))
                   if l.action == 'SELL' else
                   ((max(0, s - l.strike) if l.right == 'CE' else max(0, l.strike - s)) -
                    entry_px[(l.strike, l.right)])
                   ) * l.lots * LOT_SIZE for l in legs)
        worst = min(worst, pnl)
    return abs(worst)

def compute_charges(nc, xp, n_legs):
    brok = 20 * 2 * n_legs
    tv   = abs(nc) + abs(xp)
    stt  = 0.000625 * abs(nc)
    exc  = 0.00053  * tv
    gst  = 0.18 * (brok + exc)
    sebi = (tv / 1e7) * 10
    return round(brok + stt + exc + gst + sebi, 2)

print("Helpers ready.")


In [ ]:
_CACHE = Path('.') / 'cache'
_CACHE.mkdir(exist_ok=True)

def precompute_series(dte, strat_mod):
    """
    For each qualifying expiry: build full intraday PnL array.
    pnl_arr = nc - (piv @ wt)  [ACTUAL PnL — positive = profit, negative = loss]
    Entry restricted to Tue-Fri matching live cron schedule.
    ATM uses kite 09:25 cache as primary; falls back to nifty spot data.
    """
    series = []
    for expiry in expiry_dates:
        td = get_n_before(expiry, dte)
        if td is None or td not in all_trading_days: continue
        prev = next((e for e in expiry_dates if e < expiry), None)
        if prev and td <= prev: continue

        # Tue-Fri only — matches live cron
        if td.weekday() not in VALID_ENTRY_WDAYS: continue

        spot = get_nifty_925(td)
        if spot is None: continue
        atm  = round(spot / STRIKE_STEP) * STRIKE_STEP
        legs = get_legs(strat_mod, atm)
        if legs is None: continue

        oe = load_opt_cached(td, expiry)
        if oe is None: continue
        es = oe[oe['time_str'] == ENTRY_TIME]
        entry_px = {}; skip = False
        for l in legs:
            row = es[(es['strike_price'] == l.strike) & (es['right'] == l.right)]
            if row.empty: skip = True; break
            entry_px[(l.strike, l.right)] = float(row['open'].iloc[0])
        if skip: continue

        nc = sum(entry_px[(l.strike, l.right)] * l.lots * LOT_SIZE *
                 (1 if l.action == 'SELL' else -1) for l in legs)
        ml = compute_max_loss(legs, entry_px)
        lk = [(l.strike, l.right) for l in legs]
        wt = np.array([(1 if l.action == 'SELL' else -1) * l.lots * LOT_SIZE
                       for l in legs], dtype=float)

        pnl_vals = []
        for day in get_window(td, expiry):
            od = load_opt_cached(day, expiry)
            if od is None: continue
            ts  = ENTRY_TIME if day == td else '09:15'
            sub = od[(od['time_str'] >= ts) & (od['time_str'] <= EXIT_TIME)]
            if sub.empty: continue
            piv = sub.pivot_table(index='time_str', columns=['strike_price', 'right'],
                                  values='open', aggfunc='first')
            if any(k not in piv.columns for k in lk): continue
            # FIX: nc - (piv @ wt) = actual PnL (positive = profit)
            arr = nc - piv[lk].values @ wt
            pnl_vals.extend(arr.tolist())

        if not pnl_vals: continue

        ox = load_opt_cached(expiry, expiry)
        expiry_pnl = 0.0
        if ox is not None:
            xs = ox[ox['time_str'] == EXIT_TIME]
            p  = 0.0
            for l in legs:
                row = xs[(xs['strike_price'] == l.strike) & (xs['right'] == l.right)]
                if row.empty: continue
                e2  = float(row['open'].iloc[0])
                p  += ((entry_px[(l.strike, l.right)] - e2) if l.action == 'SELL'
                       else (e2 - entry_px[(l.strike, l.right)])) * l.lots * LOT_SIZE
            expiry_pnl = p

        series.append(dict(
            trade_date=td, expiry=expiry,
            pnl_arr=np.array(pnl_vals),
            nc=nc, ml=ml, n_legs=len(legs),
            expiry_pnl=expiry_pnl,
            charges=compute_charges(nc, expiry_pnl, len(legs)),
        ))
    return series


def apply_sltp(series, sl_pct, tp_pct):
    """Apply SL/TP to precomputed pnl_arr (actual PnL — no sign inversion)."""
    records = []
    for s in series:
        pa   = s['pnl_arr']
        sl_t = -sl_pct * s['ml']         # negative: fires when actual loss >= sl_pct * max_loss
        tp_t =  tp_pct * abs(s['nc'])    # positive: fires when actual profit >= tp_pct * credit

        shi = np.where(pa <= sl_t)[0]
        thi = np.where(pa >= tp_t)[0]
        si  = int(shi[0]) if len(shi) else len(pa)
        ti  = int(thi[0]) if len(thi) else len(pa)

        if si == len(pa) and ti == len(pa):
            xp = s['expiry_pnl']; xr = '15:20 exit'
        elif si <= ti:
            xp = float(pa[si]); xr = 'STOP LOSS'
        else:
            xp = float(pa[ti]); xr = 'TARGET HIT'

        ch = compute_charges(s['nc'], xp, s['n_legs'])
        records.append(dict(
            Trade_Date=s['trade_date'], Expiry=s['expiry'],
            N_Legs=s['n_legs'], Net_Credit=round(s['nc'], 2),
            Gross_PnL=round(xp, 2), Charges=round(ch, 2),
            Net_PnL=round(xp - ch, 2), Exit_Reason=xr,
        ))
    return pd.DataFrame(records)

print("Precompute engine ready (corrected sign).")


In [ ]:
import time

# Cache prefix 'extended_' — rebuild when data period expands (delete these to rebuild)
deployed_params = {}

for cfg in DEPLOYED:
    label   = cfg['label']
    mod     = MOD_MAP[cfg['mod_key']]
    t0      = time.time()

    series_file = _CACHE / f"extended_{label.replace(' ', '_')}_series.pkl"
    if series_file.exists():
        with open(series_file, 'rb') as f:
            series = pickle.load(f)
    else:
        series = precompute_series(cfg['dte'], mod)
        with open(series_file, 'wb') as f:
            pickle.dump(series, f)

    if not series:
        print(f"  SKIP  {label} — no valid trades"); continue

    df  = apply_sltp(series, cfg['sl'], cfg['tp'])
    wr  = (df['Net_PnL'] > 0).mean()
    net = df['Net_PnL'].sum()
    elapsed = time.time() - t0
    deployed_params[label] = (cfg['sl'], cfg['tp'], df, cfg['sname'])
    tp_n  = (df['Exit_Reason'] == 'TARGET HIT').sum()
    sl_n  = (df['Exit_Reason'] == 'STOP LOSS').sum()
    tm_n  = (df['Exit_Reason'] == '15:20 exit').sum()
    print(f"  {label:26s}  SL={cfg['sl']:.0%}  TP={cfg['tp']:.0%}  "
          f"WR={wr:.1%}  {tp_n}TP/{sl_n}SL/{tm_n}T  Net=Rs{net:>9,.0f}  ({elapsed:.1f}s)")

print(f"\nDone. {len(deployed_params)} strategies.")


In [ ]:
def simulate(df, sname, starting_capital=STARTING_CAPITAL):
    span     = SPAN[sname]
    max_lots = MAX_LOTS[sname]
    df       = df.sort_values('Trade_Date').reset_index(drop=True)
    df['Trade_Date'] = pd.to_datetime(df['Trade_Date'])

    n_legs   = int(df['N_Legs'].iloc[0])
    slip_lot = SLIP_PER_UNIT * n_legs * LOT_SIZE * 2
    eff      = span * MARGIN_BUF

    cap = float(starting_capital); peak = cap
    month = None; lots = 1; pause = 0
    equity    = [(df['Trade_Date'].iloc[0], cap)]
    trade_log = []

    for _, r in df.iterrows():
        d = r['Trade_Date']
        if cap < span:
            trade_log.append(dict(date=d, lots=0, net_pnl=0, cap=cap, reason='RUINED'))
            break
        if pause > 0:
            pause -= 1
            if pause == 0: peak = cap
            trade_log.append(dict(date=d, lots=0, net_pnl=0, cap=cap, reason='PAUSED'))
            continue
        if peak > 0 and (peak - cap) / peak >= DD_LIMIT:
            pause = PAUSE_WEEKS
            trade_log.append(dict(date=d, lots=0, net_pnl=0, cap=cap,
                                  reason=f'DD {(peak-cap)/peak:.1%}'))
            continue
        m = (d.year, d.month)
        if m != month:
            lots  = min(max_lots, max(1, int(cap // eff)))
            month = m
        gross = float(r['Gross_PnL']) * lots
        ch    = float(r['Charges'])   * lots
        slip  = slip_lot * lots
        net   = gross - ch - slip
        cap  += net
        peak  = max(peak, cap)
        equity.append((d, cap))
        trade_log.append(dict(date=d, lots=lots, net_pnl=round(net),
                              cap=round(cap), reason='TRADE'))

    log = pd.DataFrame(trade_log)
    eq  = pd.DataFrame(equity, columns=['date','capital'])
    return round(cap), log, eq

print("Simulator ready.")


In [ ]:
all_results = {}
rows = []

for label, (sl, tp, df, sname) in deployed_params.items():
    fin, log, eq = simulate(df, sname)
    trades   = log[log['reason'] == 'TRADE']
    dd_fires = log[log['reason'].str.startswith('DD', na=False)]
    wr       = (trades['net_pnl'] > 0).mean() if len(trades) else 0
    roi      = (fin - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    all_results[label] = dict(fin=fin, log=log, eq=eq, roi=roi, sl=sl, tp=tp, sname=sname)
    rows.append(dict(Label=label, SL=sl, TP=tp, WinRate=wr,
                     Trades=len(trades), DD_Fires=len(dd_fires),
                     FinalCap=fin, ROI=roi))

rdf = pd.DataFrame(rows).sort_values('FinalCap', ascending=False).reset_index(drop=True)

print()
print('=' * 95)
print(f"  10 DEPLOYED STRATEGIES — Rs{STARTING_CAPITAL:,} start | Jan 2024–Mar 2026 | Tue-Fri | kite 09:25 ATM")
print('=' * 95)
print(f"  {'Rk':<3} {'Strategy':<26} {'SL':>5} {'TP':>5} {'WR':>6} {'Trades':>7} {'DD':>3} {'Final':>12} {'ROI':>8}")
print('  ' + '-' * 83)
for i, row in rdf.iterrows():
    print(f"  {i+1:<3} {row['Label']:<26} {row['SL']:.0%}  {row['TP']:.0%} "
          f"{row['WinRate']:.1%} {int(row['Trades']):>7} {int(row['DD_Fires']):>3} "
          f"Rs{int(row['FinalCap']):>9,} {row['ROI']:>7.1f}%")


In [ ]:
for label in [row['Label'] for _, row in rdf.iterrows()]:
    r      = all_results[label]
    trades = r['log'][r['log']['reason'] == 'TRADE'].copy()
    trades['date']  = pd.to_datetime(trades['date'])
    trades['month'] = trades['date'].dt.to_period('M')

    print(f"\n--- {label}  |  SL={r['sl']:.0%}  TP={r['tp']:.0%}  "
          f"Final=Rs{r['fin']:,}  ROI={r['roi']:.1f}% ---")
    print(f"  {'Month':<8} {'Lots':>5} {'Trades':>7} {'Net PnL':>11} {'Capital':>12}")
    print('  ' + '-' * 46)
    for month, grp in trades.groupby('month'):
        print(f"  {str(month):<8} {int(grp['lots'].iloc[0]):>5} {len(grp):>7} "
              f"Rs{int(grp['net_pnl'].sum()):>9,} Rs{int(grp['cap'].iloc[-1]):>10,}")


In [ ]:
labels_ordered = [row['Label'] for _, row in rdf.iterrows()]
colors = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800',
          '#00BCD4','#8BC34A','#FF5252','#7C4DFF','#FF6D00']

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes_flat = axes.flatten()

for ax, label, color in zip(axes_flat, labels_ordered, colors):
    r  = all_results[label]
    eq = r['eq'].copy()
    eq['date'] = pd.to_datetime(eq['date'])
    ax.plot(eq['date'], eq['capital'] / 1e5, color=color, linewidth=1.8)
    ax.axhline(STARTING_CAPITAL / 1e5, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f"{label}\nSL={r['sl']:.0%} TP={r['tp']:.0%}\n"
                 f"Rs{r['fin']:,} ({r['roi']:.0f}%)", fontsize=8)
    ax.set_ylabel('Capital (Rs L)')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}L'))
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)

plt.suptitle(
    f"10 Deployed Strategies — Jan 2024–Mar 2026 | Rs{STARTING_CAPITAL:,} start | Corrected sign | Tue-Fri only | kite 09:25 ATM",
    fontsize=11)
plt.tight_layout()
plt.savefig('all10_equity_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved all10_equity_extended.png")
